<a href="https://colab.research.google.com/github/Zohaibarif69/flyrank-ml-work/blob/main/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zohaibarif69/flyrank-ml-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", HF_TOKEN is not None)

HF_TOKEN available: True


### My baseline rule

I will prioritize client-content observations that show **strong search visibility but weak click performance**, with an additional signal for **existing AI referral activity**.

The baseline score is directional decision-support, not a prediction of future performance.

**Rule:**

* Higher impressions increase priority because there is measurable search demand.
* Lower CTR relative to search visibility increases priority because the page may have an opportunity to capture more clicks.
* Existing AI referral sessions increase priority because they show current AI-search visibility.

**Reason codes:**

* `HIGH_VISIBILITY_LOW_CTR` — strong search impressions with relatively weak click-through rate.
* `AI_VISIBILITY` — existing AI referral sessions are present.
* `SEARCH_OPPORTUNITY` — meaningful search visibility is present but click performance is limited.

The action label is `REVIEW_SEARCH_OPPORTUNITY`.


In [2]:
# ML-07 — Build the baseline ranked queue

import os
import pandas as pd
import duckdb
from huggingface_hub import hf_hub_download

# Make sure HF_TOKEN already exists in Colab Secrets.
# If your earlier HF_TOKEN cell has been run, this should work.

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_path = march_file.replace("\\", "/")

df = duckdb.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    sessions_ai
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
""").df()

# Calculate CTR safely.
df["ctr"] = (
    df["gsc_clicks"] / df["gsc_impressions"].replace(0, pd.NA)
).fillna(0)

# Avoid missing values affecting the score.
df["gsc_impressions"] = df["gsc_impressions"].fillna(0)
df["gsc_avg_position"] = df["gsc_avg_position"].fillna(999)
df["sessions_ai"] = df["sessions_ai"].fillna(0)

# Simple percentile-based components.
df["visibility_score"] = df["gsc_impressions"].rank(pct=True)

df["low_ctr_score"] = (
    1 - df["ctr"].rank(pct=True)
)

df["ai_score"] = (
    df["sessions_ai"].rank(pct=True)
)

# Baseline score.
df["score"] = (
    0.50 * df["visibility_score"]
    + 0.30 * df["low_ctr_score"]
    + 0.20 * df["ai_score"]
)

# One reason code.
df["reason_code"] = "SEARCH_OPPORTUNITY"

high_visibility = df["visibility_score"] >= 0.75
low_ctr = df["low_ctr_score"] >= 0.75
has_ai = df["sessions_ai"] > 0

df.loc[high_visibility & low_ctr, "reason_code"] = "HIGH_VISIBILITY_LOW_CTR"
df.loc[has_ai & ~(high_visibility & low_ctr), "reason_code"] = "AI_VISIBILITY"

# One action label.
df["action"] = "REVIEW_SEARCH_OPPORTUNITY"

# Rank.
df = df.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = range(1, len(df) + 1)

# Select the queue columns.
queue = df[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "sessions_ai"
    ]
].copy()

# Write the required output.
output_path = "/content/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Rows ranked:", len(queue))
print("Output written:", output_path)

display(queue.head(20))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows ranked: 3611061
Output written: /content/baseline_action_score.csv


,rank,report_date,client_hash_id,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,sessions_ai
0,1,2026-03-24,client_23a62021009f63c4,content_bdf60c86117079be,0.867134,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,4195,0,0.0,31.522288,2
1,2,2026-03-29,client_23a62021009f63c4,content_4889078a4e2655ff,0.867087,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3828,0,0.0,32.590909,2
2,3,2026-03-06,client_23a62021009f63c4,content_9d1e94cbd32b0a41,0.867032,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,4371,0,0.0,33.213452,1
3,4,2026-03-07,client_23a62021009f63c4,content_5e1c049f62e33b11,0.867029,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3234,0,0.0,21.675634,4
4,5,2026-03-09,client_23a62021009f63c4,content_7d592b68cd59d19a,0.867004,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,4104,0,0.0,28.907895,1
5,6,2026-03-09,client_23a62021009f63c4,content_92a28d2e90254afa,0.867001,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,4074,0,0.0,27.885862,1
6,7,2026-03-28,client_23a62021009f63c4,content_164c1f53f13bcee1,0.866992,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3090,0,0.0,19.853074,4
7,8,2026-03-26,client_23a62021009f63c4,content_73aa61dcedebbf30,0.866988,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3980,0,0.0,51.405528,1
8,9,2026-03-12,client_23a62021009f63c4,content_b51957d7f4abe47e,0.866959,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3776,0,0.0,31.636653,1
9,10,2026-03-10,client_23a62021009f63c4,content_073221bd11582607,0.866956,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,2964,0,0.0,29.832996,4


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# ML-07 — Build ranked baseline queue

import os
import pandas as pd
import duckdb
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_path = march_file.replace("\\", "/")

df = duckdb.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    sessions_ai
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
""").df()

# Fill missing values
df["gsc_impressions"] = df["gsc_impressions"].fillna(0)
df["gsc_clicks"] = df["gsc_clicks"].fillna(0)
df["gsc_avg_position"] = df["gsc_avg_position"].fillna(999)
df["sessions_ai"] = df["sessions_ai"].fillna(0)

# CTR
df["ctr"] = (
    df["gsc_clicks"] / df["gsc_impressions"].replace(0, pd.NA)
).fillna(0)

# Percentile-based signals
df["visibility_score"] = df["gsc_impressions"].rank(pct=True)
df["low_ctr_score"] = 1 - df["ctr"].rank(pct=True)
df["ai_score"] = df["sessions_ai"].rank(pct=True)

# Baseline score
df["score"] = (
    0.50 * df["visibility_score"]
    + 0.30 * df["low_ctr_score"]
    + 0.20 * df["ai_score"]
)

# Reason code
df["reason_code"] = "SEARCH_OPPORTUNITY"

high_visibility = df["visibility_score"] >= 0.75
low_ctr = df["low_ctr_score"] >= 0.75
has_ai = df["sessions_ai"] > 0

df.loc[
    high_visibility & low_ctr,
    "reason_code"
] = "HIGH_VISIBILITY_LOW_CTR"

df.loc[
    has_ai & ~(high_visibility & low_ctr),
    "reason_code"
] = "AI_VISIBILITY"

# Action
df["action"] = "REVIEW_SEARCH_OPPORTUNITY"

# Rank
df = df.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = range(1, len(df) + 1)

queue = df[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "sessions_ai"
    ]
].copy()

print("Rows ranked:", len(queue))
display(queue.head(20))

Rows ranked: 3611061


,rank,report_date,client_hash_id,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,sessions_ai
0,1,2026-03-24,client_23a62021009f63c4,content_bdf60c86117079be,0.867134,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,4195,0,0.0,31.522288,2
1,2,2026-03-29,client_23a62021009f63c4,content_4889078a4e2655ff,0.867087,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3828,0,0.0,32.590909,2
2,3,2026-03-06,client_23a62021009f63c4,content_9d1e94cbd32b0a41,0.867032,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,4371,0,0.0,33.213452,1
3,4,2026-03-07,client_23a62021009f63c4,content_5e1c049f62e33b11,0.867029,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3234,0,0.0,21.675634,4
4,5,2026-03-09,client_23a62021009f63c4,content_7d592b68cd59d19a,0.867004,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,4104,0,0.0,28.907895,1
5,6,2026-03-09,client_23a62021009f63c4,content_92a28d2e90254afa,0.867001,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,4074,0,0.0,27.885862,1
6,7,2026-03-28,client_23a62021009f63c4,content_164c1f53f13bcee1,0.866992,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3090,0,0.0,19.853074,4
7,8,2026-03-26,client_23a62021009f63c4,content_73aa61dcedebbf30,0.866988,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3980,0,0.0,51.405528,1
8,9,2026-03-12,client_23a62021009f63c4,content_b51957d7f4abe47e,0.866959,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,3776,0,0.0,31.636653,1
9,10,2026-03-10,client_23a62021009f63c4,content_073221bd11582607,0.866956,AI_VISIBILITY,REVIEW_SEARCH_OPPORTUNITY,2964,0,0.0,29.832996,4


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# ML-07 — Top-20 review

top20 = queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda row:
        "Higher confidence: strong observed search visibility and weak CTR."
        if row["reason_code"] == "HIGH_VISIBILITY_LOW_CTR"
        else
        "Directional confidence: current AI referral activity is observed."
        if row["reason_code"] == "AI_VISIBILITY"
        else
        "Lower confidence: simple baseline signal.",
    axis=1
)

top20["what_would_make_it_wrong"] = (
    "The observed signal may be temporary, affected by query mix, "
    "tracking differences, or may not represent an actionable opportunity."
)

display(
    top20[
        [
            "rank",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
1,2,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
2,3,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
3,4,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
4,5,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
5,6,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
6,7,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
7,8,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
8,9,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."
9,10,REVIEW_SEARCH_OPPORTUNITY,AI_VISIBILITY,Directional confidence: current AI referral ac...,"The observed signal may be temporary, affected..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# ML-07 — Weak picks and leakage check

print("Weakest 10 ranked rows:")
display(
    queue.tail(10)[
        [
            "rank",
            "score",
            "reason_code",
            "action",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "sessions_ai"
        ]
    ]
)

# Confirm that no future/label fields were used.
future_or_label_fields = {
    "label",
    "target",
    "future_sessions_ai",
    "next_month_sessions_ai",
    "future_ai_sessions"
}

used_feature_fields = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_ai"
}

leaked_fields = used_feature_fields.intersection(
    future_or_label_fields
)

print("Future/label fields used:", leaked_fields)
print("Leakage check passed:", len(leaked_fields) == 0)

assert len(leaked_fields) == 0

Weakest 10 ranked rows:


,rank,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,sessions_ai
3611051,3611052,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611052,3611053,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611053,3611054,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611054,3611055,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611055,3611056,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611056,3611057,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611057,3611058,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611058,3611059,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611059,3611060,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0
3611060,3611061,0.126705,SEARCH_OPPORTUNITY,REVIEW_SEARCH_OPPORTUNITY,1,1,1.0,0


Future/label fields used: set()
Leakage check passed: True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.